In [1]:
!pip install xgboost joblib


In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

import urllib.parse
#The function of urllib.parse is to split a complex URL string into individual components so that you can easily analyze its separate parts (like the domain name, path, or query parameters).
#import urlparse


#important imports
from urllib.parse import urlparse

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib#The function of import joblib is to save your trained machine learning models to your computer's hard drive so you can reuse them later without retraining them.

from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier,RandomForestClassifier

class PhishingClassifierPipeline:
    def __init__(self):
        base_models=[
            ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)),
            ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))
        ]

        metalearner=LogisticRegression()
        self.stacking_ensemble=StackingClassifier(
            estimators=base_models,
            final_estimator=metalearner,
            cv=2,
            n_jobs=-1
        )

        self.pipeline=Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95, random_state=42)),
            ('ensemble', self.stacking_ensemble)
        ])
    def extract_features(self,url:str)->list:
        features=[]
        parsed_url=urlparse(url)
        hostname=parsed_url.netloc
        path=parsed_url.path

        #structural Lengths

        features.append(len(url))
        features.append(len(hostname))
        features.append(len(path))

        #character count
        s=['.','-','@','?','=','_','/','%','&','+','!','$']
        for i in s:
            features.append(url.count(i))
        #features.append(url.count('.'))
        #features.append(url.count('-'))
       # features.append(url.count('@'))
       # features.append(url.count('?'))
       # features.append(url.count('='))
       # features.append(url.count('_'))
       # features.append(url.count('/'))
       # features.append(url.count('%'))
       # features.append(url.count('&'))
       # features.append(url.count('+'))
       # features.append(url.count('!'))
       # features.append(url.count('$'))

        #content analysis
        features.append(sum(c.isdigit() for c in url))
        features.append(sum(c.isalpha() for c in url))
        features.append(sum(c.isdigit() for c in hostname))

        # URl Propreties
        ip_pattern = re.compile(r'(([01]?\d\d?|2[0-4]\d|25[0-5])\.){3}([01]?\d\d?|2[0-4]\d|25[0-5])')
        features.append(1 if ip_pattern.search(url) else 0)
        features.append(hostname.count('.'))
        features.append(1 if "mailto" in url else 0)
        #we can add more checks

        # Keyword Indicators
        keywords = ['login', 'verify', 'bank', 'secure', 'update', 'signin', 'account', 'free', 'bonus', 'paypal', 'amazon', 'ebay']
        url_lower=url.lower()
        for keyword in keywords:
            features.append(1 if keyword in url_lower else 0)
        return features
    #training&testing
    def fit(self,df:pd.DataFrame, url_column:str ,label_column:str):
        X=np.array([self.extract_features(url) for url in df[url_column]])
        Y=df[label_column].values

        X_train, X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)
        #stratify=y inside the train-test split. This ensures your training and testing sets have the exact same percentage of phishing vs. safe URLs, keeping your evaluation metrics stable.Step
        self.pipeline.fit(X_train,Y_train)
        train_acc=self.pipeline.score(X_train,Y_train)
        test_acc=self.pipeline.score(X_test,Y_test)
        print(f"Train Accuracy:{train_acc:.4f} | Test Accuracy: {test_acc:.4f}")
        return X_test,Y_test

    def predict_url(self,url:str)->float:
        features=np.array([self.extract_features(url)])
        return float(self.pipeline.predict_proba(features)[0,1])

    def save_model(self,filename="phishing_pipeline.joblib"):
        joblib.dump(self.pipeline,filename)

In [8]:
import os
import sys
import pandas as pd
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score
import joblib

#mport the class designed for modeling
#from model import PhishingClassifierPipeline;

def main():
    print("="*60)
    print("STARTING END-TO-END PHISHING DETECTION ML PIPELINE")
    print("="*60)

    # Data ingestion
    print("\n Ingesting Raw Data...")
    csv_file="/content/drive/MyDrive/Phishing Classifier/new_data_urls.csv"
    url_col="url"
    label_col="status"

    if not os.path.exists(csv_file):
        print(f"[-] CRITICAL ERROR : File '{csv_file}' not found in the current directory")
        print("[*]Please place your dataset CSV file here before running the pipeline.")
        sys.exit(1)

    try:
        raw_df=pd.read_csv(csv_file)
        print(f"[+] Data successsfully ingested. shape:{raw_df.shape[0]} rows,{raw_df.shape[1]} columns.")
    except Exception as e:
        print(f"[-] Critical Error:Failed to parse csv file:{e}")
        sys.exit(1)

    #Preprocess & Cleaning
    print("\n[STEP 2/5] Cleaning and Validating Data...")
    if url_col not in raw_df.columns or label_col not in raw_df.columns:
        print(f"[-] CRITICAL ERROR: Missing columns. Dataset must contain '{url_col}' and '{label_col}'.")
        print(f"[*] Available columns: {list(raw_df.columns)}")
        sys.exit(1)
    #drop rows missing critical data
    initial_rows=len(raw_df)
    clean_df=raw_df.dropna(subset=[url_col,label_col]).copy()
    dropped_nan=initial_rows-len(clean_df)
    if dropped_nan > 0:
        print(f"[!] Removed {dropped_nan} rows with missing (NaN) URL or Label values.")

    #Drop duplicate records to prevent implicit data leakage
    clean_df.drop_duplicates(subset=[url_col],inplace=True)
    dropped_dupes=initial_rows-dropped_nan-len(clean_df)

    if dropped_dupes>0:
        print(f"[!] Removed {dropped_dupes} duplicate URL records.")
    # Class balance check
    class_counts = clean_df[label_col].value_counts()
    print("[*] Target Class Distribution:")
    for cls, count in class_counts.items():
       print(f"    - Class {cls}: {count} samples ({count/len(clean_df)*100:.2f}%)")
# 3. Featur engineering & training phase
    print("\n[STEP 3/5] Starting Feature Extraction & Training Pipeline...")
    print("[*] Note: Feature extraction can take a moment depending on dataset size.")

    # Instantiate the core processing and modeling pipeline
    pipeline_manager = PhishingClassifierPipeline()

    try:
        # Extracts features, runs train_test_split, applies Scaling/PCA, and trains the stacking ensemble
        X_test, y_test = pipeline_manager.fit(clean_df, url_column=url_col, label_column=label_col)
    except Exception as e:
        print(f"[-] CRITICAL ERROR during model execution: {e}")
        sys.exit(1)
    # 4  MODEL EVALUATION PHASE (Deep Analysis)

    print("[*] Generating predictions for the evaluation subset...")
    # To run test set evaluation efficiently, we pass the pre-calculated test features through the pipeline parts
    y_pred = pipeline_manager.pipeline.predict(X_test)
    y_prob = pipeline_manager.pipeline.predict_proba(X_test)[:, 1]
    print("\n" + "-"*40)
    print("CLASSIFICATION PERFORMANCE REPORT")
    print("-"*40)
    print(classification_report(y_test, y_pred, digits=4))
    print("-"*40)
    print("CONFUSION MATRIX")
    print("-"*40)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    print(f"True Negatives (Safe URLs correctly classified): {tn}")
    print(f"False Positives (Safe URLs flagged as Phishing): {fp}")
    print(f"False Negatives (Phishing URLs missed!):        {fn}  <-- CRITICAL SECURITY RISK")
    print(f"True Positives (Phishing URLs caught):         {tp}")
    print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
    print("-"*40)

    #deployment and serialization phase
    print("\n[STEP 5/5] Serializing Pipeline for Production Deployment...")
    model_filename = "phishing_pipeline.joblib"

    try:
        pipeline_manager.save_model(model_filename)
        print(f"[+] SUCCESS: Integrated production pipeline saved as '{model_filename}'")
        print("[*] Architecture Verified: Preprocessing, Scaler, PCA, and Ensemble are locked into one single artifact.")
    except Exception as e:
        print(f"[-] Serialization Error: Failed to save the model artifact: {e}")
        sys.exit(1)

    print("\n" + "="*60)
    print("PIPELINE EXECUTION COMPLETE - READY FOR SERVER INTEGRATION")
    print("="*60)

if __name__ == "__main__":
    main()

STARTING END-TO-END PHISHING DETECTION ML PIPELINE

 Ingesting Raw Data...
[+] Data successsfully ingested. shape:822010 rows,2 columns.

[STEP 2/5] Cleaning and Validating Data...
[!] Removed 13968 duplicate URL records.
[*] Target Class Distribution:
    - Class 1: 427028 samples (52.85%)
    - Class 0: 381014 samples (47.15%)

[STEP 3/5] Starting Feature Extraction & Training Pipeline...
[*] Note: Feature extraction can take a moment depending on dataset size.
Train Accuracy:0.9377 | Test Accuracy: 0.9207
[*] Generating predictions for the evaluation subset...

----------------------------------------
CLASSIFICATION PERFORMANCE REPORT
----------------------------------------
              precision    recall  f1-score   support

           0     0.9383    0.8904    0.9137     76203
           1     0.9065    0.9477    0.9266     85406

    accuracy                         0.9207    161609
   macro avg     0.9224    0.9191    0.9202    161609
weighted avg     0.9215    0.9207    0.92